In [1]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

c:\Users\aditya\Desktop\machinelearning\nlp\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Loading the original model
tokenizer = AutoTokenizer.from_pretrained(
    "ProsusAI/finbert"
)

# Loading the trained model
model = AutoModelForSequenceClassification.from_pretrained(
    "./finbert_model"
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1587.93it/s]


In [5]:
model.eval()


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [10]:
label_map = {
    0: "negative",
    1: "neutral",
    2: "positive"
}


In [11]:
texts = [

    "Tesla stock surged after record quarterly profits",

    "Markets remained flat amid uncertainty",
    "Apple shares crashed after weak earnings guidance"
]



In [ ]:

# PREDICTIONS
for text in texts:

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():

        outputs = model(**inputs)

        prediction = torch.argmax(
            outputs.logits,
            dim=1
        ).item()

    print("\nText:")
    print(text)

    print("Predicted Sentiment:")
    print(label_map[prediction])

In [6]:
import torch
import os

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer
)

# LOADING TRAINED MODEL

model = AutoModelForSequenceClassification.from_pretrained(
    "./finbert_model"
)

model.eval()

# APPLYING DYNAMIC QUANTIZATION

quantized_model = torch.quantization.quantize_dynamic(

    model,

    {torch.nn.Linear},

    dtype=torch.qint8
)

# =====================================================
# CREATE SAVE DIRECTORY
# =====================================================

save_path = "./quantized_finbert"

os.makedirs(save_path, exist_ok=True)

# SAVING THE QUANTIZED WEIGHTS

torch.save(

    quantized_model.state_dict(),

    os.path.join(
        save_path,
        "quantized_model.pth"
    )
)

# SAVING THE CONFIG

model.config.save_pretrained(save_path)

# SAVE THE TOKENIZER

tokenizer = AutoTokenizer.from_pretrained(
    "ProsusAI/finbert"
)

tokenizer.save_pretrained(save_path)

print("Quantized model saved successfully.")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 716.63it/s]
C:\Users\aditya\AppData\Local\Temp\ipykernel_5304\1455394904.py:23: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


Quantized model saved successfully.


In [4]:

import os
model = AutoModelForSequenceClassification.from_pretrained(
    "./finbert_model"
)

model.eval()



quantized_model = torch.quantization.quantize_dynamic(

    model,

    {torch.nn.Linear},

    dtype=torch.qint8
)

save_path = "./standalone_quantized_finbert"

os.makedirs(save_path, exist_ok=True)


torch.save(

    quantized_model,

    os.path.join(
        save_path,
        "quantized_finbert_complete.pth"
    )
)

tokenizer = AutoTokenizer.from_pretrained(
    "ProsusAI/finbert"
)

tokenizer.save_pretrained(save_path)

print("Standalone quantized model saved.")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1530.83it/s]
C:\Users\aditya\AppData\Local\Temp\ipykernel_11752\805573255.py:10: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


Standalone quantized model saved.


In [8]:


# LOADING COMPLETE QUANTIZED MODEL

model = torch.load(
    "./standalone_quantized_finbert/quantized_finbert_complete.pth",
    map_location="cpu" , weights_only=False
)

model.eval()

# LOADING TOKENIZER

tokenizer = AutoTokenizer.from_pretrained(
    "./standalone_quantized_finbert"
)

print("Standalone quantized model loaded.")

Standalone quantized model loaded.


In [12]:
for text in texts:

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():

        outputs = model(**inputs)

        prediction = torch.argmax(
            outputs.logits,
            dim=1
        ).item()

    print("\nText:")
    print(text)

    print("Predicted Sentiment:")
    print(label_map[prediction])


Text:
Tesla stock surged after record quarterly profits
Predicted Sentiment:
positive

Text:
Markets remained flat amid uncertainty
Predicted Sentiment:
neutral

Text:
Apple shares crashed after weak earnings guidance
Predicted Sentiment:
negative
